# Del ADN a la proteína: extensiones con Biopython

In [172]:
from pathlib import Path
from Bio import SeqIO, Align
from Bio.Seq import Seq
from Bio.SeqUtils import seq3
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Align import substitution_matrices

## Ejercicio 1. Replicación del ADN

`Seq.complement()` sustituye cada base por su pareja (A↔T, C↔G), igual que hace la ADN polimerasa al sintetizar la hebra nueva. El resultado queda alineado base a base bajo la hebra original, es decir, leído en sentido 3'→5'. `Seq.reverse_complement()` da la misma hebra escrita en sentido 5'→3', que es la convención habitual para escribir una secuencia. Se compara con la hebra obtenida a mano.

In [173]:
hebra_superior = Seq("ATGCCGTTAGCT")
hebra_manual = "TACGGCAATCGA" 

hebra_complementaria = hebra_superior.complement()
hebra_reversa_comp = hebra_superior.reverse_complement()

print("Secuencia Original (5' - 3'):   ", hebra_superior)
print("Hebra Complementaria (3' - 5'): ", hebra_complementaria)
print("Reversa Complementaria (5' - 3'):", hebra_reversa_comp)

if str(hebra_complementaria) == hebra_manual:
    print("\nEl resultado computacional coincide con el manual.")
else:
    print("\nHay un error en las secuencias.")

Secuencia Original (5' - 3'):    ATGCCGTTAGCT
Hebra Complementaria (3' - 5'):  TACGGCAATCGA
Reversa Complementaria (5' - 3'): AGCTAACGGCAT

El resultado computacional coincide con el manual.


## Ejercicio 2. Transcripción del ADN a ARN

Se lee el gen *lacZ* de *E. coli* (β-galactosidasa) desde un FASTA. `Seq.transcribe()` solo sustituye T por U, por lo que da un ARNm correcto **únicamente si se le pasa la hebra codificante en sentido 5'→3'**. Por eso se prueba qué ocurre al cambiar la orientación de la hebra, y se comprueba además que transcribir a partir de la hebra molde da el mismo ARNm.

In [174]:
fasta_path = Path("Ej_2_lacZ_Ecoli_NC_000913.3.fasta")

record = next(SeqIO.parse(fasta_path, "fasta"))
adn = record.seq

print(f"{record.id}")
print(f"Longitud del gen: {len(adn)} nt\n")

def transcribir_y_traducir(seq, etiqueta):
    arnm = seq.transcribe()
    proteina = arnm.translate(to_stop=True)
    print(etiqueta)
    print(f"  ARNm (primeros 30 nt): 5'-{arnm[:30]}...-3'")
    print(f"  Empieza por AUG: {str(arnm[:3]) == 'AUG'}")
    print(f"  Proteína hasta el primer codón de paro: {len(proteina)} aa ({proteina[:15]}...)\n")
    return arnm

arnm = transcribir_y_traducir(adn, "1) Orientación correcta (hebra codificante 5'->3')")
transcribir_y_traducir(adn.reverse_complement(), "2) Orientación invertida (reversa complementaria)")
transcribir_y_traducir(adn.complement(), "3) Solo complementaria, sin invertir")

NC_000913.3:c366305-363231
Longitud del gen: 3075 nt

1) Orientación correcta (hebra codificante 5'->3')
  ARNm (primeros 30 nt): 5'-AUGACCAUGAUUACGGAUUCACUGGCCGUC...-3'
  Empieza por AUG: True
  Proteína hasta el primer codón de paro: 1024 aa (MTMITDSLAVVLQRR...)

2) Orientación invertida (reversa complementaria)
  ARNm (primeros 30 nt): 5'-UUAUUUUUGACACCAGACCAACUGGUAAUG...-3'
  Empieza por AUG: False
  Proteína hasta el primer codón de paro: 22 aa (LFLTPDQLVMVATGA...)

3) Solo complementaria, sin invertir
  ARNm (primeros 30 nt): 5'-UACUGGUACUAAUGCCUAAGUGACCGGCAG...-3'
  Empieza por AUG: False
  Proteína hasta el primer codón de paro: 3 aa (YWY...)



Seq('UACUGGUACUAAUGCCUAAGUGACCGGCAGCAAAAUGUUGCAGCACUGACCCUU...AUU')

Con la orientación correcta el ARNm empieza por AUG y la proteína obtenida tiene 1024 aa, la longitud de la β-galactosidasa real de *E. coli*. En las orientaciones 2 y 3 el ARNm ya no empieza por AUG (el gen empieza por ATG y termina en TAA, así que su reversa complementaria empieza por TTA y su complementaria sin invertir por TAC): se leen otros codones en otro marco de lectura y la proteína resultante no es la β-galactosidasa.

Ahora se parte de la **hebra molde**: `complement_rna()` complementa la hebra molde (3'→5') con bases de ARN, como hace la ARN polimerasa, y debe dar el mismo ARNm que la hebra codificante con T→U.

In [175]:
molde = adn.complement()             
arnm_desde_molde = molde.complement_rna()

print(f"Hebra molde (3'->5'): 3'-{molde[:30]}...-5'")
print(f"ARNm desde el molde:  5'-{arnm_desde_molde[:30]}...-3'")
print("¿Coincide con adn.transcribe()?", str(arnm_desde_molde) == str(adn.transcribe()))

Hebra molde (3'->5'): 3'-TACTGGTACTAATGCCTAAGTGACCGGCAG...-5'
ARNm desde el molde:  5'-AUGACCAUGAUUACGGAUUCACUGGCCGUC...-3'
¿Coincide con adn.transcribe()? True


## Ejercicio 3. Traducción del ARNm a proteína

`Seq.translate()` aplica la tabla estándar del código genético: lee el ARNm de tres en tres nucleótidos y marca con `*` el codón de paro. Con `to_stop=True` la traducción se corta en el primer codón de paro, que es lo que hace el ribosoma. Se compara con la traducción manual.

In [176]:
arnm = Seq("AUGUAUGCUUAA")
manual = "MYA"

completa = arnm.translate()
proteina = arnm.translate(to_stop=True)

print(f"ARNm (5'-3'):             {arnm}")
print(f"translate():              {completa}   ('*' = codón de paro)")
print(f"En tres letras:           {'-'.join(seq3(aa) for aa in proteina)}")

print("\n¿Coincide con el resultado manual (Met-Tyr-Ala)?", str(proteina) == manual)

ARNm (5'-3'):             AUGUAUGCUUAA
translate():              MYA*   ('*' = codón de paro)
En tres letras:           Met-Tyr-Ala

¿Coincide con el resultado manual (Met-Tyr-Ala)? True


## Ejercicio 4. Splicing alternativo

Se descargaron de Ensembl los CDS de todos los transcritos del gen humano FGFR2 (ENSG00000066468). Primero se traduce cada CDS y se comprueba que sea un CDS completo (longitud múltiplo de 3 y ATG inicial), porque los que no lo son pueden ser anotaciones parciales.

In [177]:
fasta_path = Path("Ej_4_ENSG00000066468.26.fasta")

transcritos = list(SeqIO.parse(fasta_path, "fasta"))
print(f"Transcritos de FGFR2 descargados de Ensembl: {len(transcritos)}\n")

proteinas = {}
print(f"{'Transcrito':<22}{'CDS (nt)':>9}  {'múltiplo de 3':<14}{'empieza en ATG':<15}{'proteína (aa)':>13}")
for r in sorted(transcritos, key=lambda r: len(r.seq)):
    seq = r.seq
    resto = len(seq) % 3
    prot = seq[:len(seq) - resto].translate(to_stop=True)
    proteinas[r.id] = prot
    print(f"{r.id:<22}{len(seq):>9}  {str(resto == 0):<14}{str(str(seq[:3]) == 'ATG'):<15}{len(prot):>13}")

Transcritos de FGFR2 descargados de Ensembl: 46

Transcrito             CDS (nt)  múltiplo de 3 empieza en ATG proteína (aa)
ENST00000985635.1           156  True          True                      51
ENST00001075071.1           159  True          True                      52
ENST00000985636.1           171  True          True                      56
ENST00000611527.2           195  True          True                      64
ENST00000613324.4           411  True          False                    136
ENST00000683250.1           414  True          True                     137
ENST00000636922.1           432  True          True                     144
ENST00000683035.1           462  True          True                     153
ENST00001075068.1           492  True          True                     163
ENST00001075069.1           723  True          True                     240
ENST00000604236.5           756  True          True                     251
ENST00000359354.7           765  True  

Varios transcritos pueden dar exactamente la misma proteína (difieren en regiones no codificantes, que no aparecen en el CDS). Para contar isoformas proteicas reales hay que comparar las secuencias de aminoácidos.

In [178]:
unicas = {}
for tid, prot in proteinas.items():
    unicas.setdefault(str(prot), []).append(tid)

print(f"Transcritos: {len(proteinas)}  |  Proteínas distintas: {len(unicas)}\n")
for secuencia, ids in unicas.items():
    if len(ids) > 1:
        print(f"Misma proteína ({len(secuencia)} aa): {', '.join(ids)}")

Transcritos: 46  |  Proteínas distintas: 40

Misma proteína (429 aa): ENST00000638709.2, ENST00000682772.1
Misma proteína (704 aa): ENST00000356226.9, ENST00000682550.2
Misma proteína (819 aa): ENST00000683211.1, ENST00000346997.6, ENST00000351936.11
Misma proteína (821 aa): ENST00000358487.10, ENST00001142304.1
Misma proteína (822 aa): ENST00000457416.7, ENST00001142305.1


Se alinea cada proteína distinta con la más larga (referencia) usando `Bio.Align.PairwiseAligner` (alineamiento global, matriz BLOSUM62, sin penalizar los huecos en los extremos para no castigar a las isoformas truncadas). Para cada isoforma se calcula qué fracción de la referencia cubre y qué identidad tiene en la parte alineada.

In [179]:
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5
aligner.end_gap_score = 0

def comparar(ref, iso):
    aln = aligner.align(ref, iso)[0]
    identicos = alineados = 0
    for (r0, r1), (i0, i1) in zip(*aln.aligned):
        identicos += sum(a == b for a, b in zip(ref[r0:r1], iso[i0:i1]))
        alineados += r1 - r0
    return identicos, alineados

referencia = max(unicas, key=len)
print(f"Referencia: {unicas[referencia][0]} ({len(referencia)} aa)\n")
print(f"{'Transcrito':<22}{'aa':>5}{'cobertura ref.':>16}{'identidad':>11}{'aa sin pareja':>15}")
for secuencia in sorted(unicas, key=len, reverse=True):
    identicos, alineados = comparar(referencia, secuencia)
    print(f"{unicas[secuencia][0]:<22}{len(secuencia):>5}{alineados/len(referencia)*100:>15.1f}%"
          f"{identicos/alineados*100:>10.1f}%{len(secuencia)-alineados:>15}")

Referencia: ENST00000457416.7 (822 aa)

Transcrito               aa  cobertura ref.  identidad  aa sin pareja
ENST00000457416.7       822          100.0%     100.0%              0
ENST00000358487.10      821           99.6%      97.3%              2
ENST00000985634.1       820           99.8%     100.0%              0
ENST00000683211.1       819           99.4%      97.3%              2
ENST00000937093.2       813           98.7%      97.3%              2
ENST00001044441.1       802           97.3%      98.0%              2
ENST00001044442.1       786           95.6%      99.6%              0
ENST00001044438.1       785           95.3%      96.8%              2
ENST00000369056.6       769           93.6%      99.9%              0
ENST00000369058.7       768           93.4%     100.0%              0
ENST00001075072.1       766           92.9%      97.0%              2
ENST00000613048.5       732           88.8%      97.0%              2
ENST00001044439.1       731           88.9%     10

Isoformas de **igual longitud pero secuencia distinta** son un caso especial: indican que se ha cambiado un segmento por otro sin ganar ni perder residuos. Se localiza dónde difieren.

In [180]:
por_longitud = {}
for secuencia, ids in unicas.items():
    por_longitud.setdefault(len(secuencia), []).append((secuencia, ids[0]))

hay_pares = False
for longitud, grupo in sorted(por_longitud.items()):
    if len(grupo) > 1:
        hay_pares = True
        (a, id_a), (b, id_b) = grupo[0], grupo[1]
        difs = [i + 1 for i, (x, y) in enumerate(zip(a, b)) if x != y]
        print(f"{id_a} vs {id_b} ({longitud} aa): {len(difs)} sustituciones entre las posiciones {difs[0]} y {difs[-1]}")
if not hay_pares:
    print("No hay isoformas distintas con la misma longitud.")

ENST00000369059.6 vs ENST00000357555.9 (707 aa): 626 sustituciones entre las posiciones 37 y 707


**Cómo pueden afectar estas diferencias a la función de la proteína.** La comparación muestra dos patrones:

- **Isoformas más cortas que la referencia, con la parte que falta sin pareja en el alineamiento.** Pierden segmentos enteros, que según la región pueden ser dominios extracelulares de tipo inmunoglobulina, la región transmembrana o el dominio quinasa. Una isoforma sin dominio quinasa o sin región transmembrana no puede transmitir la señal del receptor como la isoforma completa. Los CDS muy cortos o cuya longitud no es múltiplo de 3 pueden ser anotaciones parciales, por lo que se interpretan con cautela.
- **Isoformas de la misma longitud con sustituciones concentradas en una región.** Es lo esperable en exones mutuamente excluyentes. Es el caso descrito en la literatura para FGFR2-IIIb (epitelial) y FGFR2-IIIc (mesenquimal), que difieren en el tercer dominio de tipo inmunoglobulina y, con ello, en qué ligandos FGF reconocen. Para saber qué transcrito corresponde a cada una hay que comparar con las secuencias de UniProt (P21802).

## Ejercicio 5. Introducción a las proteínas

Se usa la hemoglobina humana (PDB 1GZX). Primero se analiza la secuencia de cada cadena a partir del FASTA del PDB con `ProteinAnalysis` (extremos N y C, peso molecular y porcentaje de residuos hidrofóbicos).

In [181]:
fasta_path = Path("Ej_5_rcsb_pdb_1GZX.fasta")

for record in SeqIO.parse(fasta_path, "fasta"):
    seq = str(record.seq)
    analisis = ProteinAnalysis(seq)
    hidrofobicos_pct = sum(seq.count(aa) for aa in "AVLIMFWY") / len(seq) * 100

    print(f"{record.id}")
    print(f"Longitud: {len(seq)} aa")
    print(f"Extremo N-terminal: {seq[0]}  |  Extremo C-terminal: {seq[-1]}")
    print(f"Peso molecular: {analisis.molecular_weight():.1f} Da")
    print(f"% de aminoácidos hidrofóbicos: {hidrofobicos_pct:.1f}%")
    print("-" * 60)

1GZX_2|Chains
Longitud: 146 aa
Extremo N-terminal: V  |  Extremo C-terminal: H
Peso molecular: 15867.0 Da
% de aminoácidos hidrofóbicos: 44.5%
------------------------------------------------------------
1GZX_1|Chains
Longitud: 141 aa
Extremo N-terminal: V  |  Extremo C-terminal: R
Peso molecular: 15126.2 Da
% de aminoácidos hidrofóbicos: 46.1%
------------------------------------------------------------


Casi la mitad de los residuos de ambas cadenas de la hemoglobina (44,5% y 46,1%) son hidrofóbicos, típico de una proteína globular cuyo núcleo debe empaquetarse evitando el contacto con el agua. Esto respalda la reflexión teórica: sustituir uno de esos residuos internos por uno hidrofílico rompería ese empaquetamiento. De hecho, la mutación causante de la anemia falciforme (Glu6Val en la cadena β) sigue esta misma lógica, aunque en un residuo de superficie en vez de uno interno.

## Ejercicio 6. Actividad integradora: del ADN a la proteína

El pipeline encadena replicación, transcripción y traducción, e informa de cada paso con mensajes `[Etapa]`. Antes de empezar comprueba que el CDS es válido (longitud múltiplo de 3, ATG inicial, codón de paro final) y emite un `[AVISO]` si algo no cuadra. Cada etapa reproduce el proceso biológico:

- **Replicación:** cada hebra parental sirve de molde, de modo que se obtienen dos moléculas hijas, cada una con una hebra parental y otra nueva.
- **Transcripción:** el ARNm se sintetiza leyendo la hebra molde (`complement_rna()`) y se comprueba que coincide con la hebra codificante con T→U.
- **Traducción:** `translate()` avisa si hay un codón de paro prematuro o si falta el codón de paro.

In [182]:
CODONES_PARO_ADN = {"TAA", "TAG", "TGA"}

def pipeline_dogma_central(adn: Seq, nombre="secuencia", n=30):
    def corto(s):
        return f"{s[:n]}..." if len(s) > n else str(s)

    # Comprobaciones previas
    print(f"[ADN] {nombre}: hebra codificante de {len(adn)} nt")
    print(f"[ADN]   5'-{corto(adn)}-3'")
    if len(adn) % 3 != 0:
        print(f"[AVISO] La longitud ({len(adn)} nt) no es múltiplo de 3: el último codón está incompleto.")
    if str(adn[:3]) != "ATG":
        print(f"[AVISO] La secuencia no empieza por ATG (empieza por {adn[:3]}).")
    if str(adn[-3:]) not in CODONES_PARO_ADN:
        print(f"[AVISO] La secuencia no termina en un codón de paro (termina en {adn[-3:]}).")

    # Replicación
    molde = adn.complement()
    print("[Replicación] La helicasa separa las hebras parentales:")
    print(f"[Replicación]   codificante 5'-{corto(adn)}-3'")
    print(f"[Replicación]   molde       3'-{corto(molde)}-5'")
    nueva_1 = adn.complement()
    nueva_2 = adn
    print("[Replicación] La ADN polimerasa sintetiza dos hebras nuevas (replicación semiconservativa):")
    print(f"[Replicación]   Hija 1 = parental codificante + nueva 3'-{corto(nueva_1)}-5'")
    print(f"[Replicación]   Hija 2 = parental molde       + nueva 5'-{corto(nueva_2)}-3'")
    print(f"[Replicación] ¿Las dos moléculas hijas son idénticas a la original? "
          f"{str(nueva_1.complement()) == str(adn) and str(nueva_2.complement()) == str(molde)}")

    # Transcripción
    arnm = molde.complement_rna()
    print("[Transcripción] La ARN polimerasa lee la hebra molde 3'->5' y sintetiza en 5'->3' (U en lugar de T):")
    print(f"[Transcripción]   ARNm 5'-{corto(arnm)}-3'")
    print(f"[Transcripción] ¿Coincide con la hebra codificante con T->U? {str(arnm) == str(adn.transcribe())}")

    # Traducción
    resto = len(arnm) % 3
    if resto:
        print(f"[AVISO] Se ignoran los últimos {resto} nt, que no forman un codón completo.")
    arnm_codones = arnm[:len(arnm) - resto]
    completa = str(arnm_codones.translate())
    proteina = arnm_codones.translate(to_stop=True)
    print(f"[Traducción] Codón de inicio: {arnm[:3]} ({seq3(str(arnm_codones.translate())[0])})")
    if "*" not in completa:
        print("[AVISO] No hay codón de paro: la traducción llega hasta el final del ARNm.")
    elif completa.index("*") < len(completa) - 1:
        print(f"[AVISO] Codón de paro prematuro en el codón {completa.index('*') + 1}: la proteína queda truncada.")
    else:
        print("[Traducción] Codón de paro encontrado al final: la traducción termina correctamente.")
    print(f"[Traducción] Proteína de {len(proteina)} aa: {corto(proteina)}")
    return proteina

fasta_path = Path("Ej_6_ENSG00000141510.21.fasta")

registros = {r.id: r for r in SeqIO.parse(fasta_path, "fasta")}
id_canonico = "ENST00000269305.9"
record = registros.get(id_canonico) or next(iter(registros.values()))
print(f"Transcrito usado: {record.id}\n")

_ = pipeline_dogma_central(record.seq, nombre="TP53")

Transcrito usado: ENST00000269305.9

[ADN] TP53: hebra codificante de 1182 nt
[ADN]   5'-ATGGAGGAGCCGCAGTCAGATCCTAGCGTC...-3'
[Replicación] La helicasa separa las hebras parentales:
[Replicación]   codificante 5'-ATGGAGGAGCCGCAGTCAGATCCTAGCGTC...-3'
[Replicación]   molde       3'-TACCTCCTCGGCGTCAGTCTAGGATCGCAG...-5'
[Replicación] La ADN polimerasa sintetiza dos hebras nuevas (replicación semiconservativa):
[Replicación]   Hija 1 = parental codificante + nueva 3'-TACCTCCTCGGCGTCAGTCTAGGATCGCAG...-5'
[Replicación]   Hija 2 = parental molde       + nueva 5'-ATGGAGGAGCCGCAGTCAGATCCTAGCGTC...-3'
[Replicación] ¿Las dos moléculas hijas son idénticas a la original? True
[Transcripción] La ARN polimerasa lee la hebra molde 3'->5' y sintetiza en 5'->3' (U en lugar de T):
[Transcripción]   ARNm 5'-AUGGAGGAGCCGCAGUCAGAUCCUAGCGUC...-3'
[Transcripción] ¿Coincide con la hebra codificante con T->U? True
[Traducción] Codón de inicio: AUG (Met)
[Traducción] Codón de paro encontrado al final: la traducci

El resultado final (393 aa) coincide con la longitud real conocida de la proteína p53 humana.